# Paper figures

Run this notebook from the repository root (or from `notebooks/`; the setup cell finds the repo). Figures are written as `.png` and `.svg` under `results/figures/`.

1. **NCH ΔAUPRC heatmap** — age band × history horizon, `AUPRC(dtr_age_temporal_new) − AUPRC(dtr_temporal_only_new)`.
2. **Synthetic age × lag heatmaps** — mean `P̂(Y | a, τ)` for Oracle, DTR, temporal-only, and CEHR-BERT, plus residuals against the oracle. Probability panels share one color scale; residual panels share one diverging scale centered at 0.
3. **History-horizon curve** — the same ΔAUPRC against `30d, 90d, 180d, 1y, 3y, 5y, Full`.

The NCH table is cached in `analysis/nch/cache/`. The first full test-split pass takes several minutes on CPU. Later cells reuse that cache unless `recompute=True`.

In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, display

REPO = Path.cwd().resolve()
if not (REPO / "analysis").is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(REPO)

## NCH metrics

Held-out test windows. For each horizon, input events older than that cutoff (relative to the last input timestamp) are dropped. Labels are unchanged. `Full` is the untruncated window. Micro-AUPRC uses a 5e6 label–score reservoir when a cell is larger than that, the same cap as the baseline evaluator.

In [ ]:
from analysis.nch.evaluate import get_metrics

nch = get_metrics(recompute=False, max_examples=0, batch_size=128, device="cpu")
print("examples", nch["n_examples"], "elapsed_s", round(nch["elapsed_s"], 1))

## NCH heatmap

Rows are age bands `<1`, `1–5`, `6–11`, `12–17`. Columns are `30d`, `90d`, `180d`, `1y`, `3y`, `Full`.

In [ ]:
from analysis.nch.plot_delta_auprc_heatmap import plot as plot_nch_heatmap

heatmap = plot_nch_heatmap(nch)
display(Image(filename=str(heatmap["png"])))

## History horizon

Overall ΔAUPRC (not stratified by age) as the available history grows, including `5y`.

In [ ]:
from analysis.nch.plot_history_horizon import plot as plot_horizon

horizon = plot_horizon(nch)
display(Image(filename=str(horizon["png"])))

## Synthetic surfaces (S2)

Axes are identical across panels: lag on x, current age on y. Color on Oracle / DTR / temporal-only / CEHR-BERT is the mean target probability and uses one shared range taken from all four grids. Residuals use one shared diverging range, `P̂_model − P_oracle`, centered at 0. Set `recompute=True` to rebuild the grids from checkpoints instead of the stored forward-pass cache.

In [ ]:
from analysis.synthetic.plot_dtr_heatmap import plot as plot_dtr_surface
from analysis.synthetic.plot_cehrbert_heatmap import plot as plot_cehr_surface

dtr_surface = plot_dtr_surface(scenario="S2", recompute=False)
cehr_surface = plot_cehr_surface(scenario="S2", recompute=False)
print(
    "shared P limits",
    round(dtr_surface["vmin"], 4),
    round(dtr_surface["vmax"], 4),
    "shared |residual|",
    round(dtr_surface["residual_limit"], 4),
)
display(Image(filename=str(dtr_surface["png"])))
display(Image(filename=str(cehr_surface["png"])))